In [0]:
table_name = "fact_pharmacy"
silver_table_fqn = f"he_prod_incen_analytics_dbw_01.silver.{table_name}"

df = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.fact_pharmacy")

In [0]:
from pyspark.sql.functions import col, upper, trim, when, to_date, row_number, abs as spark_abs
from pyspark.sql.window import Window

# 1. Enrich: Map Source SKs -> Business Keys
df_map_member = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_members").select(col("member_sk").cast("int").alias("src_member_sk"), col("member_id"))
df_map_provider = spark.read.table("he_prod_incen_analytics_dbw_01.bronze.dim_providers").select(col("provider_sk").cast("int").alias("src_provider_sk"), col("provider_id"))

df = df.join(df_map_member, df["member_sk"] == df_map_member["src_member_sk"], "left").drop("src_member_sk", "member_sk")
df = df.join(df_map_provider, df["provider_sk"] == df_map_provider["src_provider_sk"], "left").drop("src_provider_sk", "provider_sk")

# 2. Standardize, Cast Dates/Ints/Decimals/Booleans
for c in ["pharmacy_claim_id", "member_id", "provider_id", "pharmacy_id", "ndc_code", "drug_name", "drug_strength", "therapeutic_class"]:
    df = df.withColumn(c, upper(trim(col(c))))

df = df.withColumn("fill_date", to_date(trim(col("fill_date"))))

for int_col in ["quantity_dispensed", "days_supply", "refill_count"]:
    df = df.withColumn(int_col, trim(col(int_col)).cast("int"))

for dec_col in ["ingredient_cost", "dispensing_fee", "gross_cost", "insurance_paid", "member_copay", "member_coinsurance", "manufacturer_discount"]:
    is_valid = trim(col(dec_col)).rlike("^[+-]?([0-9]+\\.?[0-9]*|[0-9]*\\.?[0-9]+)$")
    df = df.withColumn(dec_col, when(is_valid, trim(col(dec_col)).cast("decimal(15,2)")).otherwise(None))
    df = df.withColumn(dec_col, spark_abs(col(dec_col)))

for bool_col in ["generic_flag", "prior_auth_required", "step_therapy_fail", "specialty_pharmacy", "drug_allergy_flag", "duplicate_therapy_flag", "hmo_excluded_flag"]:
    df = df.withColumn(bool_col, when(upper(trim(col(bool_col))).isin("YES", "1", "TRUE"), True).when(upper(trim(col(bool_col))).isin("NO", "0", "FALSE"), False).otherwise(None).cast("boolean"))

dedup_window = Window.partitionBy("pharmacy_claim_id").orderBy(col("_ingested_at").desc())
df = df.withColumn("_dq_is_deduped", row_number().over(dedup_window)).filter(col("_dq_is_deduped") == 1).drop("_dq_is_deduped", "_ingested_at", "_source_file", "pharmacy_sk")

display(df.limit(5))

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table_fqn)
print(f"✅ Successfully wrote {table_name} to Silver layer.")